# Protótipo: histograma da diferença entre vetores

A cada instante, as variáveis formam um vetor de dimensão `N`:

\[
\mathbf{y}_t = [y_{t,1}, \ldots, y_{t,N}], \qquad
\hat{\mathbf{y}}_t = [\hat{y}_{t,1}, \ldots, \hat{y}_{t,N}]
\]

A opção `normalize_vectors` controla se **cada vetor real e predito é normalizado individualmente pela própria norma L2 antes da subtração**:

\[
\mathbf{u}_t = \frac{\mathbf{y}_t}{\|\mathbf{y}_t\|_2},
\qquad
\hat{\mathbf{u}}_t = \frac{\hat{\mathbf{y}}_t}{\|\hat{\mathbf{y}}_t\|_2}
\]

\[
d_t = \|\mathbf{u}_t - \hat{\mathbf{u}}_t\|_2
\]

Quando `normalize_vectors=False`, calcula-se diretamente:

\[
d_t = \|\mathbf{y}_t - \hat{\mathbf{y}}_t\|_2
\]

Portanto, cada instante contribui com **um único valor não negativo** para o histograma. Vetores nulos são mantidos como vetores nulos durante a normalização (evita divisão por zero).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
class VectorDifferenceHistograms:
    """Histogram of L2 differences between multivariate real/predicted vectors.

    Expected input shape: (n_windows, window_size, n_signals).
    At each window/time position, the last axis is treated as one N-dimensional
    vector. If normalize_vectors=True, each real and predicted vector is
    independently divided by its own L2 norm before their difference is taken.
    Zero vectors remain zero during normalization.
    """

    def __init__(
        self,
        ground_truth,
        prediction,
        signal_names=None,
        normalize_vectors=False,
    ):
        self.ground_truth = np.asarray(ground_truth, dtype=np.float64)
        self.prediction = np.asarray(prediction, dtype=np.float64)
        self.signal_names = (
            None if signal_names is None else list(signal_names)
        )
        self.normalize_vectors = bool(normalize_vectors)

        self._validate_inputs()
        self.vector_differences = self._calculate_vector_differences()
        self.distances = np.linalg.norm(self.vector_differences, axis=-1)
        self.flat_distances = self.distances.ravel()

    def _validate_inputs(self):
        if self.ground_truth.ndim != 3:
            raise ValueError(
                "ground_truth must have shape (n_windows, window_size, n_signals)."
            )
        if self.prediction.shape != self.ground_truth.shape:
            raise ValueError("ground_truth and prediction must have the same shape.")
        if not np.isfinite(self.ground_truth).all():
            raise ValueError("ground_truth contains non-finite values.")
        if not np.isfinite(self.prediction).all():
            raise ValueError("prediction contains non-finite values.")
        if self.signal_names is not None:
            if len(self.signal_names) != self.ground_truth.shape[-1]:
                raise ValueError("signal_names must contain one name per signal.")
            if len(set(self.signal_names)) != len(self.signal_names):
                raise ValueError("signal_names must be unique.")

    @staticmethod
    def _normalize_vectors(vectors):
        norms = np.linalg.norm(vectors, axis=-1, keepdims=True)
        # A zero vector remains zero; nonzero vectors become unit vectors.
        return np.divide(
            vectors,
            norms,
            out=np.zeros_like(vectors),
            where=norms > 0,
        )

    def _calculate_vector_differences(self):
        real_vectors = self.ground_truth
        predicted_vectors = self.prediction

        if self.normalize_vectors:
            real_vectors = self._normalize_vectors(real_vectors)
            predicted_vectors = self._normalize_vectors(predicted_vectors)

        return real_vectors - predicted_vectors

    def statistics(self):
        """Return descriptive statistics of the vector-difference magnitudes."""
        return pd.Series(self.flat_distances, name="vector_difference_norm").describe()

    def plot(
        self,
        bins=100,
        figsize=(10, 5),
        show=True,
        save_path=None,
        log_scale=False,
        normalize_histogram=False,
        x_range=None,
        y_range=None,
        title=None,
        font_size=12,
    ):
        """Plot the histogram of vector-difference magnitudes.

        normalize_histogram controls relative frequencies and is independent
        of normalize_vectors, which controls vector normalization.
        """
        if bins < 1:
            raise ValueError("bins must be a positive integer.")
        if self.flat_distances.size == 0:
            raise ValueError("There are no vector distances to plot.")

        values = self.flat_distances
        if x_range is not None:
            xmin, xmax = x_range
            if xmin < 0 or xmax <= xmin:
                raise ValueError("x_range must satisfy 0 <= min < max.")
            values_for_range = values[(values >= xmin) & (values <= xmax)]
            if values_for_range.size == 0:
                raise ValueError("No distances fall inside x_range.")
            hist_range = (xmin, xmax)
        else:
            hist_range = (0.0, float(values.max()) if values.max() > 0 else 1.0)

        weights = None
        if normalize_histogram:
            weights = np.full(values.shape, 1.0 / values.size)

        fig, ax = plt.subplots(figsize=figsize)
        ax.hist(values, bins=bins, range=hist_range, weights=weights, edgecolor="black")
        if log_scale:
            ax.set_yscale("log")
        if y_range is not None:
            ax.set_ylim(*y_range)

        ax.set_xlabel("Vector Difference Magnitude (L2)", fontsize=font_size)
        ax.set_ylabel(
            "Relative Frequency" if normalize_histogram else "Frequency",
            fontsize=font_size,
        )
        if title is None:
            mode = "Normalized Vectors" if self.normalize_vectors else "Original Scale"
            title = f"Vector Difference Histogram — {mode}"
        ax.set_title(title, fontsize=font_size)
        ax.tick_params(axis="both", labelsize=font_size)
        ax.grid(True, alpha=0.25)

        if save_path is not None:
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, bbox_inches="tight", dpi=150)
        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig, ax


## Testes unitários do comportamento matemático

Os testes abaixo verificam:
1. sem normalização, a distância é a norma L2 da subtração original;
2. com normalização, vetores de mesma direção têm distância zero mesmo com magnitudes diferentes;
3. vetores ortogonais unitários têm distância `sqrt(2)`;
4. vetores nulos não geram `NaN` ou divisão por zero;
5. entradas inválidas são rejeitadas.

In [ ]:
# 1) Sem normalização: ||[3, 4] - [0, 0]|| = 5
real = np.array([[[3.0, 4.0]]])
pred = np.array([[[0.0, 0.0]]])
model = VectorDifferenceHistograms(real, pred, normalize_vectors=False)
assert np.allclose(model.flat_distances, [5.0])

# 2) Com normalização: [3, 4] e [6, 8] apontam na mesma direção -> distância 0
real = np.array([[[3.0, 4.0]]])
pred = np.array([[[6.0, 8.0]]])
model = VectorDifferenceHistograms(real, pred, normalize_vectors=True)
assert np.allclose(model.flat_distances, [0.0], atol=1e-12)

# 3) Vetores ortogonais: distância entre [1, 0] e [0, 1] = sqrt(2)
real = np.array([[[1.0, 0.0]]])
pred = np.array([[[0.0, 1.0]]])
model = VectorDifferenceHistograms(real, pred, normalize_vectors=True)
assert np.allclose(model.flat_distances, [np.sqrt(2)])

# 4) Vetores nulos: zero permanece zero, sem NaN
real = np.array([[[0.0, 0.0], [1.0, 0.0]]])
pred = np.array([[[0.0, 0.0], [0.0, 0.0]]])
model = VectorDifferenceHistograms(real, pred, normalize_vectors=True)
assert np.allclose(model.flat_distances, [0.0, 1.0])
assert np.isfinite(model.flat_distances).all()

print("Todos os testes matemáticos passaram.")

In [ ]:
# 5) Validação de formatos e valores
def must_raise(callable_, error=ValueError):
    try:
        callable_()
    except error:
        return
    raise AssertionError(f"Expected {error.__name__} was not raised.")

must_raise(lambda: VectorDifferenceHistograms(
    np.zeros((2, 3)), np.zeros((2, 3))
))
must_raise(lambda: VectorDifferenceHistograms(
    np.zeros((1, 2, 2)), np.zeros((1, 2, 3))
))
must_raise(lambda: VectorDifferenceHistograms(
    np.array([[[np.nan, 0.0]]]), np.zeros((1, 1, 2))
))
must_raise(lambda: VectorDifferenceHistograms(
    np.zeros((1, 1, 2)), np.zeros((1, 1, 2)),
    signal_names=["A"]
))

print("Todos os testes de validação passaram.")

## Exemplo sintético e comparação das duas opções

Neste exemplo, são gerados vetores reais e preditos com pequenas diferenças. Os dois histogramas usam exatamente os mesmos dados, alterando apenas `normalize_vectors`.

In [ ]:
rng = np.random.default_rng(42)
n_windows, window_size, n_signals = 8, 120, 4

y_true = rng.normal(size=(n_windows, window_size, n_signals))
# Introduz uma pequena perturbação nas previsões
y_pred = y_true + rng.normal(scale=0.15, size=y_true.shape)

raw_model = VectorDifferenceHistograms(
    y_true, y_pred, normalize_vectors=False
)
unit_model = VectorDifferenceHistograms(
    y_true, y_pred, normalize_vectors=True
)

print("Estatísticas — vetores na escala original:")
display(raw_model.statistics())
print("Estatísticas — vetores normalizados individualmente:")
display(unit_model.statistics())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(raw_model.flat_distances, bins=40, edgecolor="black")
axes[0].set_title("Sem normalização dos vetores")
axes[0].set_xlabel("||y - y_pred||₂")
axes[0].set_ylabel("Frequência")

axes[1].hist(unit_model.flat_distances, bins=40, edgecolor="black")
axes[1].set_title("Com normalização dos vetores")
axes[1].set_xlabel("||y/||y||₂ - y_pred/||y_pred||₂||₂")
axes[1].set_ylabel("Frequência")

for ax in axes:
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Exemplo de uso do método `plot`

`normalize_vectors` define a geometria dos vetores antes da diferença. `normalize_histogram` é outra opção: apenas transforma as contagens do histograma em frequências relativas.

In [ ]:
unit_model.plot(
    bins=50,
    normalize_histogram=True,
    log_scale=False,
    show=True,
)